# Рецепт дообучения: нулевая точка и этапы A–E

Продолжение [`kaggle_phase_d0_commit.ipynb`](kaggle_phase_d0_commit.ipynb).
Трудные негативы и оценки учителя уже посчитаны — здесь они подключаются
входом и не пересчитываются.

Тетрадь рассчитана на **Save Version → Save & Run All (Commit)**.

**Входы:** датасеты `ru-fin-retrieval` и `ru-fin-queries`, модели
`intfloat/multilingual-e5-small` и `BAAI/bge-reranker-v2-m3`, вывод
ядра с подготовкой (`ru-fin-d0`).

**Что считается.** Сначала нулевая точка — dev необученного ученика тем же
протоколом, которым меряются все модели проекта. Затем этапы рецепта:
контрастив и три длительности обучения, трудные негативы, дистилляция
и её сочетание с контрастивом, матрёшка, подбор числа эпох и температуры.
После каждого этапа считается dev и выносится вердикт.

**Теста здесь нет и быть не может.** Шаг `train` фильтрует запросы
по составу dev и без dev отказывается работать. Прогон по всему набору —
отдельный шаг, и он делается один раз, после того как рецепт зафиксирован.

**Если прогон оборвётся, потеряно будет мало.** Журнал сохраняется после
каждого этапа, а продолжить рецепт можно по одному журналу: каждый этап
обучается с исходных весов, поэтому веса отклонённых этапов не нужны
вовсе — они и убираются по ходу.

In [ ]:
import os
# Хаб запрещён, карта одна. Ставится до первого импорта torch
# и transformers, иначе библиотеки успеют прочитать настройки.
for ключ in ('HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_DATASETS_OFFLINE'):
    os.environ[ключ] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print('офлайн и одна карта')

In [ ]:
# ──────── НАСТРОЙКИ ПРОГОНА ────────
ТЕГИ = []          # пусто — весь рецепт; иначе список меток, например ['a-1ep']
БАТЧ_ОЦЕНКИ = 128  # размер батча при подсчёте эмбеддингов, не при обучении

import time
НАЧАЛО = time.time()


def прошло() -> str:
    с = time.time() - НАЧАЛО
    return f'{с // 3600:.0f} ч {(с % 3600) // 60:02.0f} мин'

## 1. Карта и библиотеки

Обучение держит в памяти градиенты и состояния оптимизатора сверх весов,
поэтому карта нужна обязательно. `sentence-transformers` не ниже 3.0:
в более ранних нет `SentenceTransformerTrainer`, на котором держится
обучение двумя наборами данных сразу.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch, transformers, sentence_transformers
print('transformers', transformers.__version__)
print('sentence-transformers', sentence_transformers.__version__)
print('карт:', torch.cuda.device_count())
assert torch.cuda.is_available(), 'нет карты: Settings -> Accelerator'

# На Turing нет bf16, и обучение идёт в fp32 — это решение рецепта,
# а не умолчание библиотеки. Печатается для записи в отчёт.
print('bf16 поддерживается:', torch.cuda.is_bf16_supported())

from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers.losses import (CachedMultipleNegativesRankingLoss,
                                          MatryoshkaLoss)
import datasets, accelerate
print('datasets', datasets.__version__, '| accelerate', accelerate.__version__)
print('всё, что нужно обучению, на месте')

## 2. Код, данные и подготовка на место

In [ ]:
import glob, os, shutil, zipfile


def find_source(root='/kaggle/input', max_depth=5):
    found, archives = [], []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs:
            stale = ('__notebook__.ipynb' in files or '__output__.json' in files)
            found.append((1 if stale else 0, cur))
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    if found:
        found.sort()
        return found[0][1], None
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


src, archive = find_source()
assert src, 'датасет с кодом не подключён'
print('код:', src)
if archive:
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for имя in os.listdir(src):
        путь, цель = os.path.join(src, имя), os.path.join('/kaggle/working', имя)
        if os.path.isdir(путь):
            shutil.copytree(путь, цель, dirs_exist_ok=True)
        elif имя.endswith(('.py', '.md')):
            shutil.copyfile(путь, цель)

# Каноническая нарезка: пересчитывать её нельзя, границы фрагмента зависят
# от версии токенизатора, а идентификатор позиционный — эталон съехал бы молча.
dst = '/kaggle/working/chunks/base.jsonl'
if os.path.exists(dst):
    os.remove(dst)
найдено = (glob.glob('/kaggle/input/**/chunks_base.jsonl', recursive=True)
           + glob.glob('/kaggle/input/**/chunks_base.jsonl.gz', recursive=True))
assert найдено, 'нет chunks_base.jsonl'
os.makedirs('/kaggle/working/chunks', exist_ok=True)
import gzip
opener = gzip.open if найдено[0].endswith('.gz') else open
with opener(найдено[0], 'rb') as fi, open(dst, 'wb') as fo:
    shutil.copyfileobj(fi, fo, length=1 << 20)
n = sum(1 for _ in open(dst, encoding='utf-8'))
assert n == 62594, f'ожидалось 62594, а не {n}: нарезка не каноническая'
print('нарезка:', n, 'фрагментов')

# Подготовка D0. Без неё этапы B и далее учиться не на чем.
подготовка = glob.glob('/kaggle/input/**/train_prepared.jsonl', recursive=True)
assert подготовка, ('нет train_prepared.jsonl: подключите вывод ядра '
                    'с подготовкой через + Add Input -> Your Work')
print('подготовка:', подготовка[0],
      sum(1 for _ in open(подготовка[0], encoding='utf-8')), 'записей')

# Журнал прежнего прогона, если он был: посчитанные этапы не повторяются.
os.makedirs('/kaggle/working/train', exist_ok=True)
прежний = glob.glob('/kaggle/input/**/journal.json', recursive=True)
if прежний:
    shutil.copyfile(прежний[0], '/kaggle/working/train/journal.json')
    import json
    было = json.load(open(прежний[0], encoding='utf-8'))
    print('журнал прежнего прогона:',
          [э['tag'] for э in было.get('этапы', [])])
else:
    print('журнала нет: рецепт считается с нуля')
print('\nпрошло', прошло())

## 3. Нулевая точка и рецепт

Один процесс на всё: корпус, сплит и разметка читаются один раз, а шагов
в рецепте дюжина. Веса держатся только у лучшего принятого этапа —
остальные не нужны, потому что каждый этап обучается с исходных.

In [ ]:
import subprocess, sys

cmd = ['python', 'run_phase_d.py', '--step', 'all',
       '--prepared', подготовка[0],
       '--batch-size', str(БАТЧ_ОЦЕНКИ),
       '--keep-only-best']
if ТЕГИ:
    cmd += ['--tags'] + list(ТЕГИ)
print('запуск:', ' '.join(cmd), flush=True)
код = subprocess.run(cmd, cwd='/kaggle/working').returncode
print('\nкод возврата:', код, '| прошло', прошло())

## 4. Таблица рецепта

In [ ]:
import json, os, sys
sys.path.insert(0, '/kaggle/working')
from rufin.training.journal import Journal

путь = '/kaggle/working/train/journal.json'
if os.path.exists(путь):
    j = Journal.load(путь)
    print(j.table())
    лучшее, базовая = j.best(), j.baseline()
    if лучшее and базовая and лучшее.tag != базовая.tag:
        print(f'\nлучшее принятое: {лучшее.tag}, dev NDCG@10 {лучшее.ndcg:.3f} '
              f'против {базовая.ndcg:.3f} у необученного ученика '
              f'({лучшее.ndcg - базовая.ndcg:+.3f})')
    print('\nэтапов в журнале:', len(j.entries))
else:
    print('журнала нет: ни один этап не досчитался')

# Кэш нарезки в вывод не едет: он большой и получается из корпуса заново.
import shutil
shutil.rmtree('/kaggle/working/chunks', ignore_errors=True)
размер = sum(os.path.getsize(os.path.join(б, и))
             for б, _, ф in os.walk('/kaggle/working') for и in ф)
print(f'вывод версии: {размер / 1073741824:.2f} ГБ')